# 1. Reproduce the manuscript from saved fits

This notebook reconstructs the numerical displays for *Same Metric, Different Models: Interpreting Political Positions* without running the optimizer.

It checks the exact construction, verifies the saved Senate fits against the bundled data, displays the reported scores, and redraws the figures. The inputs in `data/` and `reference/` are preserved. New figures are written to `outputs/reference-figures/`.

Install the dependencies described in the repository README, select that Python environment as the notebook kernel, and run the cells in order. Plotting uses Matplotlib and its bundled fonts. No LaTeX installation, network access, or external dataset download is needed.

To study the preprocessing, continue to `2_model_and_data.ipynb`. To estimate new fits, use `3_refit_senate.ipynb`.


In [ ]:
from pathlib import Path
import json
import os
import sys

# Works when Jupyter starts in the repository root or a directory below it.
working_directory = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (working_directory, *working_directory.parents)
     if (candidate / "idealpoint").is_dir()
     and (candidate / "reference").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the replication repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / "data"
REFERENCE = ROOT / "reference"
os.environ.setdefault("MPLBACKEND", "Agg")
cache_directory = ROOT / "outputs" / "_matplotlib"
cache_directory.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(cache_directory))

import numpy as np
import pandas as pd
from IPython.display import Image, JSON, Markdown, display

print("Repository:", ROOT.name)


## 1. Check the exact example

Three actors and three items occupy the same locations on a line. Replacing distance by squared distance changes the centered log-odds matrix. The exact arithmetic checks concern this construction and its one-dimensional implications. They are independent of the Senate fits.


In [ ]:
from idealpoint.checks import exact_checks, verify_results
from idealpoint.figures import render_all

exact_report = exact_checks()
display(JSON(exact_report, expanded=False))
assert exact_report["passed"], "An exact arithmetic check failed."


## 2. Verify the saved empirical results

The verifier reconstructs quantities from the saved data and parameter vectors. Inspect the report for the separate internal-consistency and reference-comparison results. A successful verification checks reproducibility of the stored calculations. It does not certify that an optimizer found a stationary point or a global optimum.


In [ ]:
verification = verify_results(REFERENCE, DATA, compare_reference=True)
display(JSON(verification, expanded=False))
assert verification["passed"], "Saved-result consistency checks failed; inspect the report."


## 3. Inspect the sample and score table

The analysis concerns Yea/Nay votes in the 114th U.S. Senate, using one frozen split. Held-out entries belong to senators and roll calls represented in training. They do not test predictions for new senators or new roll calls.

The coordinate prior standard deviation is varied while the additive-effect prior and distance coefficient remain fixed. Equal parameter priors under the two exponents induce different priors on vote probabilities.


In [ ]:
design = json.loads((DATA / "fit_comparison_design.json").read_text())
sample = {
    "Senators": design["n_members"],
    "Roll calls": design["n_rollcalls"],
    "Observed Yea/Nay votes": design["n_observed"],
    "Training votes": design["n_train"],
    "Held-out votes": design["n_test"],
    "Missing entries": design["n_missing"],
}
display(pd.Series(sample, name="Count").to_frame())

scores = pd.read_csv(REFERENCE / "fit_comparison_results_scores.csv")
saved_results = json.loads((REFERENCE / "fit_comparison_results.json").read_text())
scores["training_log_loss"] = [
    saved_results["sensitivity"][str(float(row.coordinate_sd))][str(int(row.q))]
    ["training"]["negative_log_likelihood"]
    for row in scores.itertuples()
]
score_table = scores.loc[:, [
    "coordinate_sd", "q", "training_log_loss", "negative_log_likelihood", "brier", "accuracy"
]].copy()
score_table["q"] = score_table["q"].map({1: "Unsquared", 2: "Squared"})
score_table = score_table.rename(columns={
    "coordinate_sd": "Coordinate prior SD",
    "q": "Distance term",
    "training_log_loss": "Training log loss",
    "negative_log_likelihood": "Held-out log loss",
    "brier": "Held-out Brier score",
    "accuracy": "Held-out accuracy",
})
display(score_table.round(6))


## 4. Inspect optimization diagnostics

These are approximate joint MAP fits selected by the training posterior objective, not by held-out performance. The unsquared norm is nondifferentiable when an actor and item coincide. A solver's success flag, or a small change in its objective, does not establish stationarity in this setting.

The table below reports diagnostics for the selected fits. The full record of starts and refinements remains in `reference/fit_comparison_results_starts.csv`.


In [ ]:
selected_diagnostics = scores.loc[:, [
    "coordinate_sd", "q", "objective", "optimizer_success_flag",
    "max_abs_selected_gradient"
]].copy()
display(selected_diagnostics)

saved_results = json.loads((REFERENCE / "fit_comparison_results.json").read_text())
if "optimization_caution" in saved_results:
    display(JSON({"optimization_caution": saved_results["optimization_caution"]}, expanded=False))


## 5. Regenerate the figures

The two main figures show the exact centered-logit construction and the Senate illustration. The supplementary figure gives the fuller fit comparison. The portable redraws retain the underlying results but use a bundled font, so their typography can differ from the original manuscript exports in `reference/manuscript_figures/`.


In [ ]:
FIGURES = ROOT / "outputs" / "reference-figures"
figure_paths = render_all(REFERENCE, DATA, FIGURES)
figure_titles = {
    "centered_logits": "Figure 1. Exact centered-logit construction",
    "senate_illustration": "Figure 2. Senate illustration",
    "fit_comparison": "Supplementary figure. Detailed fit comparison",
}
for name, title in figure_titles.items():
    display(Markdown(f"### {title}"))
    display(Image(filename=str(figure_paths[name]["png"])))
    print("PDF:", Path(figure_paths[name]["pdf"]).relative_to(ROOT))


## What this reproduction establishes

The figures and tables reproduce the saved analysis. They show that greater party separation in these fitted maps accompanies higher held-out log loss under unsquared distance. The loss gap changes with the coordinate prior.

This is a descriptive comparison of fitted procedures on one split, with unresolved convergence for the unsquared fits. It does not establish a general ranking of the two model families or replicate the published Bayesian estimators of Lee et al. The metricity argument and the exact one-dimensional result do not depend on this empirical comparison.
